# Phase 3 – Apache Spark and Big Data Dataset

This notebook contains the Spark implementation for Phase 3 of the Big Data project.

- Part B.1–B.2: Apache Spark / SparkSQL – Florjetë Kuka
- Part B.3: New Big Data Dataset – Amela Syla

## Part B.1–B.2: Apache Spark / SparkSQL  
**Author: Florjetë Kuka**

This section imports the datasets from Phase 1 and Phase 2 into Apache Spark, converts them to Parquet format, and adapts two XPath/XQuery queries from Phase 2 into SparkSQL.

In [0]:
worldbank_df = spark.table("world_bank_dataset")
fuel_df = spark.table("fuel_price")
xml_df = spark.table("worlbank_dwh")

### Verification of Imported Datasets

After importing the datasets, the first rows, schemas, and row counts were checked to verify that the data was loaded correctly.

In [0]:
worldbank_df.show(5)
fuel_df.show(5)
xml_df.show(5)

+-----------+-----------+----+--------------------+--------------+-----+
|    Country|CountryCode|Year|       IndicatorName|IndicatorValue|RowId|
+-----------+-----------+----+--------------------+--------------+-----+
|Afghanistan|        AFG|2015|GDP growth (annua...|   1.451314661|    1|
|Afghanistan|        AFG|2015|GDP per capita (c...|   565.5697304|    2|
|Afghanistan|        AFG|2015|Inflation, consum...|  -0.661709165|    3|
|Afghanistan|        AFG|2015|Urban population ...|   24.46491184|    4|
|Afghanistan|        AFG|2015|    Trade (% of GDP)|          NULL|    5|
+-----------+-----------+----+--------------------+--------------+-----+
only showing top 5 rows
+----------+-------------+-------------+--------------------+-----+-----+
|      Date|      Country|       Region|            FuelType|Price|RowId|
+----------+-------------+-------------+--------------------+-----+-----+
|2017-01-09|United States|North America|Tax_Rate_on_Fuel (%)| 18.0|32280|
|2017-01-16|United Stat

In [0]:
worldbank_df.printSchema()
fuel_df.printSchema()
xml_df.printSchema()

root
 |-- Country: string (nullable = true)
 |-- CountryCode: string (nullable = true)
 |-- Year: long (nullable = true)
 |-- IndicatorName: string (nullable = true)
 |-- IndicatorValue: string (nullable = true)
 |-- RowId: long (nullable = true)

root
 |-- Date: date (nullable = true)
 |-- Country: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- FuelType: string (nullable = true)
 |-- Price: double (nullable = true)
 |-- RowId: long (nullable = true)

root
 |-- CountryName: string (nullable = true)
 |-- IndicatorName: string (nullable = true)
 |-- IndicatorValue: double (nullable = true)
 |-- Year: long (nullable = true)



In [0]:
print("World Bank rows:", worldbank_df.count())
print("Fuel Price rows:", fuel_df.count())
print("WorldBank DWH rows:", xml_df.count())

World Bank rows: 26334
Fuel Price rows: 35160
WorldBank DWH rows: 26334


### Converting Datasets to Parquet Format

The imported datasets were converted into Parquet format, which is optimized for Apache Spark processing.

In [0]:
worldbank_df.write.mode("overwrite").parquet("/FileStore/parquet/worldbank_parquet")

fuel_df.write.mode("overwrite").parquet("/FileStore/parquet/fuelprices_parquet")

xml_df.write.mode("overwrite").parquet("/FileStore/parquet/worldbank_xml_parquet")

In [0]:
worldbank_parquet = spark.read.parquet("/FileStore/parquet/worldbank_parquet")

fuel_parquet = spark.read.parquet("/FileStore/parquet/fuelprices_parquet")

xml_parquet = spark.read.parquet("/FileStore/parquet/worldbank_xml_parquet")

In [0]:
worldbank_parquet.createOrReplaceTempView("worldbank")
fuel_parquet.createOrReplaceTempView("fuelprices")
xml_parquet.createOrReplaceTempView("worldbank_xml")

### Query 1 – Average Indicator Values for Kosovo

This query adapts the XPath/XQuery logic from Phase 2 into SparkSQL. It filters the data for Kosovo and calculates the average value of the indicators using `WHERE`, `AVG`, and `GROUP BY`.

In [0]:
spark.sql("""
SELECT
    CountryName,
    AVG(CAST(IndicatorValue AS DOUBLE)) AS avg_value
FROM worldbank_xml
WHERE CountryName = 'Kosovo'
GROUP BY CountryName
""").show()

+-----------+-----------------+
|CountryName|        avg_value|
+-----------+-----------------+
|     Kosovo|713.6297971428573|
+-----------+-----------------+



### Query 2 – Ranking Countries by GDP per Capita

This query adapts the XPath/XQuery logic from Phase 2 into SparkSQL. It filters the GDP per capita indicator and ranks countries by its value using `WHERE`, `ORDER BY`, and `LIMIT`.

In [0]:
spark.sql("""
SELECT
    CountryName,
    Year,
    CAST(IndicatorValue AS DOUBLE) AS gdp_per_capita
FROM worldbank_xml
WHERE IndicatorName = 'GDP per capita (current US$)'
AND IndicatorValue IS NOT NULL
ORDER BY gdp_per_capita DESC
LIMIT 10
""").show()

+-------------+----+--------------+
|  CountryName|Year|gdp_per_capita|
+-------------+----+--------------+
|       Monaco|2024|   288001.4334|
|       Monaco|2023|   256799.7886|
|       Monaco|2022|   226052.0019|
|       Monaco|2021|    223823.364|
|Liechtenstein|2023|   206780.5904|
|Liechtenstein|2021|   201944.8303|
|       Monaco|2019|   193746.7856|
|       Monaco|2018|   188298.3157|
|Liechtenstein|2022|   188055.0032|
|       Monaco|2020|   176891.8865|
+-------------+----+--------------+



## Part B.3: New Big Data Dataset  
**Author: Amela Syla**

This section imports and analyzes the new Big Data dataset `world_oil_data`. The dataset contains more than 500,000 rows and is used to demonstrate Spark processing through row counting, summary statistics, aggregations, SparkSQL queries.

In [0]:
from pyspark.sql.functions import col, sum as _sum, to_date, year

In [0]:
big_df = spark.table("world_oil_data")

### Dataset Verification

After importing the dataset, the first rows, schema, and row count are examined to verify that the data has been loaded correctly.

In [0]:
print("=== FIRST ROWS ===")
big_df.show(5)

print("=== SCHEMA ===")
big_df.printSchema()

print("=== ROW COUNT ===")
print(big_df.count())

=== FIRST ROWS ===
+--------+-------+--------------------+--------------------+---------------+----------+---------------+
|    Year|Country| Energy Product Name|      Flow Breakdown|Unit Of Measure|     Value|Assessment Code|
+--------+-------+--------------------+--------------------+---------------+----------+---------------+
|6/1/2010| Kuwait|Liquefied petrole...|             Exports|            KBD|     129.0|              3|
|6/1/2010| Kuwait|Liquefied petrole...|             Exports|          KTONS|333.620697|              3|
|6/1/2010| Kuwait|Liquefied petrole...|             Imports|             KL|       0.0|              3|
|7/1/2010| Kuwait|Liquefied petrole...|IProducts transfe...|        CONVBBL|   11600.0|              3|
|7/1/2010| Kuwait|Liquefied petrole...|IProducts transfe...|            KBD|       0.0|              3|
+--------+-------+--------------------+--------------------+---------------+----------+---------------+
only showing top 5 rows
=== SCHEMA ===
root
 

### Summary Statistics

Apache Spark provides descriptive statistics that help understand the distribution and characteristics of the dataset.

In [0]:
print("=== SUMMARY STATISTICS ===")
big_df.describe().show()

=== SUMMARY STATISTICS ===
+-------+--------+-------+-------------------+--------------+---------------+------------------+------------------+
|summary|    Year|Country|Energy Product Name|Flow Breakdown|Unit Of Measure|             Value|   Assessment Code|
+-------+--------+-------+-------------------+--------------+---------------+------------------+------------------+
|  count|  504926| 504926|             504926|        504926|         504926|            504926|            504926|
|   mean|    NULL|   NULL|               NULL|          NULL|           NULL| 5499.345397109477|2.5475020101955534|
| stddev|    NULL|   NULL|               NULL|          NULL|           NULL|25372.627623533208|0.8289319131401508|
|    min|1/1/2002|Algeria|          Crude Oil|Closing stocks|        CONVBBL|               0.0|                 1|
|    max|9/1/2024|  Yemen| Total oil products| Stock changes|          KTONS|         1204561.0|                 3|
+-------+--------+-------+-------------------

### Aggregation by Country

The dataset is grouped by country and the total energy value is calculated for each country.

In [0]:
country_stats = big_df.groupBy("Country") \
    .agg(_sum("Value").alias("Total_Energy")) \
    .orderBy(col("Total_Energy").desc())

country_stats.show(10)

+-------------+--------------------+
|      Country|        Total_Energy|
+-------------+--------------------+
|United States| 6.266370708778055E8|
| Saudi Arabia|3.5438467041975063E8|
|        China|2.0229227545453855E8|
|       Russia| 1.768312802803137E8|
|       Mexico|  1.54347305451489E8|
|       Canada|1.2877878222270088E8|
|       Norway|1.2234416597362146E8|
|        India|1.2091987380665836E8|
|       Kuwait|1.1379648238120772E8|
|        Qatar| 9.757494554229662E7|
+-------------+--------------------+
only showing top 10 rows


### Aggregation by Energy Product

The dataset is grouped by energy product in order to identify the products with the highest total energy values.

In [0]:
product_stats = big_df.groupBy("Energy Product Name") \
    .agg(_sum("Value").alias("Total_Value")) \
    .orderBy(col("Total_Value").desc())

product_stats.show(10)

+--------------------+--------------------+
| Energy Product Name|         Total_Value|
+--------------------+--------------------+
|           Crude Oil| 5.665313430713375E8|
|  Total oil products|4.2218352206985795E8|
|               Total|3.9199660227875775E8|
|Motor and aviatio...|2.2298110078838202E8|
|      Gas/diesel oil|2.0211959308492088E8|
|Liquefied petrole...|2.0011054527619204E8|
|  Other oil products|1.4012685395029017E8|
|           Kerosenes|1.3279932843123569E8|
|            Fuel Oil|1.3106599270022148E8|
|                 NGL|1.1727403868211538E8|
+--------------------+--------------------+
only showing top 10 rows


### SparkSQL Analysis

A SparkSQL query is executed to validate the aggregation results and demonstrate SQL-based analysis on the dataset.

In [0]:
big_df.createOrReplaceTempView("energy")

sql_result = spark.sql("""
SELECT Country,
       SUM(Value) AS Total_Energy
FROM energy
GROUP BY Country
ORDER BY Total_Energy DESC
LIMIT 10
""")

sql_result.show()

+-------------+--------------------+
|      Country|        Total_Energy|
+-------------+--------------------+
|United States| 6.266370708778055E8|
| Saudi Arabia|3.5438467041975063E8|
|        China|2.0229227545453855E8|
|       Russia| 1.768312802803137E8|
|       Mexico|  1.54347305451489E8|
|       Canada|1.2877878222270088E8|
|       Norway|1.2234416597362146E8|
|        India|1.2091987380665836E8|
|       Kuwait|1.1379648238120772E8|
|        Qatar| 9.757494554229662E7|
+-------------+--------------------+

